# Phase 1 — **LoRA fine-tune** (mHuBERT-147) · on-the-fly

The weighted-sum experiment answered the question "which frozen layers carry the ASR signal" (9 and 10 dominate, val-CER 4.4%). This notebook tries a **different axis**: does **adapting the backbone with LoRA** help, rather than merely *reading* it.

**The key difference from weighted-sum:** there is **no feature cache**. The LoRA adapters are updated at every step, so the backbone output is not fixed and cannot be precomputed. Every epoch runs a **forward and backward** pass through the backbone over the raw audio, which makes epochs **heavier** than in the frozen-head run, since the backbone now takes part in backpropagation.

- **LoRA:** rank-16 adapters on `q_proj` and `v_proj` (the base weights stay frozen).
- **Head:** again [9,10] weighted-sum plus CTC, so it stays comparable with the weighted-sum runs.
- **bf16 autocast** (Blackwell/RTX PRO 6000 native; no GradScaler needed).
- ReduceLR (`thr=0.005 rel`) plus delta-checkpoint early stopping (0.5% relative, patience 12), the same as the attached notebook.
- When it finishes it saves to Drive and **shuts the session down automatically** (last cell).

## 1. Drive and dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.11" jiwer torchaudio soundfile

## 2. Import & config

In [ ]:
import os, json, time, gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, HubertModel
from peft import LoraConfig, get_peft_model, TaskType
import jiwer

# ================= CONFIG =================
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
BACKBONE_ID   = "utter-project/mHuBERT-147"
DATASET_ID    = "openslr/librispeech_asr"
SR            = 16_000
HID           = 768

# --- HEAD: weighted-sum, comparable with the weighted-sum runs ---
WS_LAYERS     = [9, 10]           # 9 and 10 came out dominant, the head blends them
N_WS          = len(WS_LAYERS)

# --- LoRA ---
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.0               # deterministic features, for a clean comparison with weighted-sum
LORA_TARGETS  = ["q_proj", "v_proj"]

# --- PATH (LoRA adapter + head; features are NEVER written to disk) ---
OUTPUT_DIR    = "/content/drive/MyDrive/CLEAR/phase1_lora_9-10"

MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES  = None

TRAIN_BATCH        = 32           # you can raise this to 64 on 96 GB of VRAM
ACCUMULATION_STEPS = 8            # eff. batch = TRAIN_BATCH * ACCUM = 256
NUM_EPOCHS         = 50           # LoRA adapts fast and can overfit, early stopping will cut it
HEAD_LR            = 1e-3         # the head learns faster
LORA_LR            = 2e-4         # adapters want a lower LR
LR_PATIENCE        = 4
LR_FACTOR          = 0.5
STOP_PATIENCE      = 12
NUM_WORKERS        = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
print(f"[ENV] torch={torch.__version__} device={DEVICE}"
      + (f" gpu={torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else ""))
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Vocab and tokenizer (byte-for-byte identical to the baseline)

In [ ]:
chars = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")
vocab = {c: i for i, c in enumerate(chars)}
vocab["|"]     = len(vocab)
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)
with open("vocab.json", "w") as f:
    json.dump(vocab, f)
tokenizer = Wav2Vec2CTCTokenizer("vocab.json", unk_token="[UNK]",
                                 pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SR, padding_value=0.0,
    do_normalize=True, return_attention_mask=True)
VOCAB_SIZE = len(vocab)
BLANK_ID   = vocab["[PAD]"]
UNK_ID     = vocab["[UNK]"]
ID2CH      = {i: c for c, i in vocab.items()}
print(f"[VOCAB] {VOCAB_SIZE} token, blank={BLANK_ID}")

## 4. On-the-fly dataset + collate (raw audio; no cache)

In [ ]:
def _load_split(split, max_samples):
    if split == "train.100":
        hf = load_dataset(DATASET_ID, data_files={"train": "clean/train.100/*.parquet"},
                          split="train", verification_mode="no_checks")
    else:
        hf = load_dataset(DATASET_ID, data_files={"validation": "clean/validation/*.parquet"},
                          split="validation", verification_mode="no_checks")
    hf = hf.cast_column("audio", Audio(sampling_rate=SR))
    if max_samples: hf = hf.select(range(min(max_samples, len(hf))))
    return hf

class OnTheFlyDataset(Dataset):
    def __init__(self, hf_ds):
        self.ds = hf_ds
        self.labels = [tokenizer(ex["text"].upper()).input_ids for ex in hf_ds]  # tokenised in advance
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        audio = np.asarray(self.ds[i]["audio"]["array"], dtype=np.float32)
        return torch.from_numpy(audio), torch.tensor(self.labels[i], dtype=torch.long), i

def on_the_fly_collate(items):
    audios, labs, idxs = zip(*items)
    B = len(audios)
    x_audio = torch.nn.utils.rnn.pad_sequence(audios, batch_first=True, padding_value=0.0)
    audio_lens = torch.tensor([len(a) for a in audios], dtype=torch.long)
    Smax = max(len(l) for l in labs)
    y = torch.full((B, Smax), BLANK_ID, dtype=torch.long)
    for i, l in enumerate(labs): y[i, :len(l)] = l
    return x_audio, y, audio_lens, torch.tensor([len(l) for l in labs], dtype=torch.long), list(idxs)

train_ds = OnTheFlyDataset(_load_split("train.100", MAX_TRAIN_SAMPLES))
dev_ds   = OnTheFlyDataset(_load_split("validation", MAX_EVAL_SAMPLES))

train_dl = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True,
                      collate_fn=on_the_fly_collate, num_workers=NUM_WORKERS,
                      pin_memory=False, prefetch_factor=2, persistent_workers=True)
dev_dl   = DataLoader(dev_ds, batch_size=TRAIN_BATCH, shuffle=False,
                      collate_fn=on_the_fly_collate, num_workers=NUM_WORKERS,
                      pin_memory=False, prefetch_factor=2, persistent_workers=True)
print(f"[DATA] train={len(train_ds)} dev={len(dev_ds)} | {len(train_dl)} train batch/epoch")

## 5. Backbone + LoRA + head
The backbone stays in `eval()` mode (SpecAugment and dropout off, so the features are deterministic and the comparison with weighted-sum stays clean) **but** the adapter parameters have `requires_grad=True`, and because the forward runs **outside** `no_grad` the gradients reach the adapters.

In [ ]:
# --- load the backbone and wrap it in LoRA (base frozen, adapters trainable) ---
print(f"[LORA] loading the backbone ({BACKBONE_ID})...")
raw_backbone = HubertModel.from_pretrained(BACKBONE_ID).to(DEVICE)
feat_len_fn  = raw_backbone._get_feat_extract_output_lengths      # do not trust the peft proxy, keep the reference

lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                      target_modules=LORA_TARGETS, bias="none",
                      task_type=TaskType.FEATURE_EXTRACTION)
backbone = get_peft_model(raw_backbone, lora_cfg)
backbone.eval()                     # deterministic features, the adapters still receive gradients
backbone.print_trainable_parameters()

class WeightedSumHead(nn.Module):
    def __init__(self, n_ws=N_WS, dim=HID, vocab_size=VOCAB_SIZE):
        super().__init__()
        self.layer_w = nn.Parameter(torch.zeros(n_ws))          # start out equal
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.ELU(), nn.Linear(dim, vocab_size))
    def forward(self, x):                    # x: [B, T, N_WS, 768]
        w = self.layer_w.softmax(0)
        feat = (x * w[None, None, :, None]).sum(2)
        return self.net(feat)

from itertools import groupby
def greedy_decode(ids):
    dec = [ID2CH.get(k, "") for k, _ in groupby(ids) if k != BLANK_ID and k != UNK_ID]
    return "".join(dec).replace("|", " ").strip()

@torch.no_grad()
def evaluate_wer(model, dl, ds):
    model.eval(); backbone.eval()
    hyps, refs = [], []
    for x_audio, y, x_audio_lens, ylen, idxs in dl:
        x_audio = x_audio.to(DEVICE, non_blocking=True)
        am = torch.zeros(x_audio.shape, dtype=torch.long, device=DEVICE)
        for i, L in enumerate(x_audio_lens): am[i, :L] = 1
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = backbone(x_audio, attention_mask=am, output_hidden_states=True)
            hs = torch.stack([out.hidden_states[L] for L in WS_LAYERS], dim=2)
        xlen = feat_len_fn(x_audio_lens.to(DEVICE))
        logits = model(hs.float())
        pred = logits.argmax(-1).cpu().numpy()
        for b, i in enumerate(idxs):
            hyps.append(greedy_decode(pred[b, :xlen[b]].tolist()))
            refs.append(ds.ds[i]["text"].upper())
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps), hyps, refs

## 6. Training (LoRA and head) with ReduceLR and delta-checkpoint early stopping
Critical: the backbone forward runs **outside** `no_grad`. Two parameter groups, the head (`HEAD_LR`) and LoRA (`LORA_LR`).

In [ ]:
model = WeightedSumHead().to(DEVICE)

lora_params = [p for p in backbone.parameters() if p.requires_grad]
head_params = list(model.parameters())
n_lora = sum(p.numel() for p in lora_params)
n_head = sum(p.numel() for p in head_params)
print(f"[MODEL] trainable: head={n_head:,} + lora={n_lora:,} = {n_head+n_lora:,}  "
      f"(targets={LORA_TARGETS}, r={LORA_R}, layers={WS_LAYERS})")

optimizer = torch.optim.AdamW(
    [{"params": head_params, "lr": HEAD_LR},
     {"params": lora_params, "lr": LORA_LR}], fused=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE,
    threshold=0.005, threshold_mode="rel")
ctc = nn.CTCLoss(blank=BLANK_ID, reduction="mean", zero_infinity=True)

best_cer, best_epoch = float("inf"), 0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train(); backbone.eval()          # the adapters receive gradients, the features stay deterministic
    t0, tot_loss, nb = time.perf_counter(), 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    for x_audio, y, x_audio_lens, ylen, idxs in train_dl:
        x_audio, y = x_audio.to(DEVICE), y.to(DEVICE)
        am = torch.zeros(x_audio.shape, dtype=torch.long, device=DEVICE)
        for i, L in enumerate(x_audio_lens): am[i, :L] = 1

        # --- backbone forward with GRAD ON, so the LoRA adapters learn ---
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            out = backbone(x_audio, attention_mask=am, output_hidden_states=True)
            hs = torch.stack([out.hidden_states[L] for L in WS_LAYERS], dim=2)
        xlen = feat_len_fn(x_audio_lens.to(DEVICE))

        logits = model(hs.float())                       # head fp32
        logp = logits.log_softmax(-1).transpose(0, 1)
        loss = ctc(logp, y, xlen, ylen.to(DEVICE)) / ACCUMULATION_STEPS
        loss.backward()

        if ((nb + 1) % ACCUMULATION_STEPS == 0) or ((nb + 1) == len(train_dl)):
            optimizer.step(); optimizer.zero_grad(set_to_none=True)
        tot_loss += loss.item() * ACCUMULATION_STEPS; nb += 1

    va_wer, va_cer, _, _ = evaluate_wer(model, dev_dl, dev_ds)
    lr_head = optimizer.param_groups[0]["lr"]; lr_lora = optimizer.param_groups[1]["lr"]
    wnow = model.layer_w.softmax(0).detach().cpu().numpy().round(3)
    print(f"epoch {epoch:>3} | loss {tot_loss/nb:.3f} | {time.perf_counter()-t0:.1f}s "
          f"| VAL wer {va_wer*100:.1f}% cer {va_cer*100:.1f}% "
          f"| lr(h/l) {lr_head:.1e}/{lr_lora:.1e} | w={wnow}")

    scheduler.step(va_cer)

    # --- delta checkpoint, save only on a 0.5% relative improvement ---
    if va_cer < best_cer * (1.0 - 0.005):
        best_cer, best_epoch = va_cer, epoch
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "ctc_head_lora.pt"))
        backbone.save_pretrained(os.path.join(OUTPUT_DIR, "lora_adapter"))
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"   [SAVE] new best val-CER {va_cer*100:.2f}% -> {OUTPUT_DIR}")
    elif epoch - best_epoch >= STOP_PATIENCE:
        print(f"[STOP] no improvement above 0.5% for {STOP_PATIENCE} epochs. "
              f"(best {best_cer*100:.2f}% @ e{best_epoch})")
        break

print(f"[DONE] best val-CER {best_cer*100:.2f}% @ epoch {best_epoch}")
print(f"[COMPARE] baseline (layer 9 only) 8.2% | weighted-sum [9,10]/[8,9,10] about 4.4% | LoRA above.")
TRAINING_DONE = True

## 7. Shut the session down automatically
Run this **after training has finished and the best model has been written to Drive**. It flushes Drive and releases the Colab runtime, which stops billing. Outside Colab the process simply exits.

In [ ]:
# shut down only if training finished, this prevents an accidental early run
assert globals().get("TRAINING_DONE", False), "Do not run this cell before training has finished."
import time
print("[SHUTDOWN] syncing the checkpoints to Drive...")
time.sleep(15)                                   # headroom for Drive's async sync
try:
    drive.flush_and_unmount(); print("[SHUTDOWN] Drive flushed.")
except Exception as e:
    print(f"[SHUTDOWN] Drive flush skipped: {e}")
try:
    from google.colab import runtime
    print("[SHUTDOWN] releasing the Colab runtime...")
    runtime.unassign()
except Exception as e:
    print(f"[SHUTDOWN] not Colab ({e}), shutting the process down.")
    import os; os._exit(0)